In [ ]:
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

In [ ]:
from IPython.utils import text
base_url='https://books.toscrape.com'
base_data = requests.get('https://books.toscrape.com')
bs=BeautifulSoup(base_data.text,'html.parser')
categories_links=bs.find_all('a')[3:-1]
categories={}

for category_link in categories_links:
  if category_link.string and 'books' in category_link.get('href'):
    # print(category_link.get('href'), category_link.string.strip())
    categories[category_link.string.strip()]=category_link.get('href')

# for category_link,category in categories.items():
#   print(category_link,'--',category)

data=[]

books=[['name','price','rating','availability','category']]
for category,category_link in categories.items():
  data=requests.get(base_url+'/'+category_link)
  bs=BeautifulSoup(data.text,'html.parser')
  for element in bs.find_all('article',class_='product_pod'):
    book_name=element.h3.a.text
    book_price=element.find('p', class_='price_color').text
    book_availability=element.find('p', class_='instock availability').text.strip()
    book_rating=element.find('p', class_='star-rating').attrs['class'][1].strip()
    # print('Book name is ', book_name)
    # print('Book price is ', book_price)
    # print('Book rating is ', book_rating)
    # print('Book availability is ', book_availability)
    # print('Book category is ', category)
    books.append([book_name,book_price,book_rating,book_availability,category])

In [ ]:
import csv

# for book in books:
#   print(book)

#write data to file
with open('data.csv','w', newline='',encoding='utf-8') as f:
  writer=csv.writer(f)
  writer.writerows(books)

#convert file data to dataframe
df=pd.read_csv('data.csv')
print(df.head(10))

In [ ]:
##################################################################################################################################
#converting price to float
df['price_gbp']=pd.to_numeric(df['price'].str.split('Â£').str[1])

#Converting rating to integer
rating_mapping={'One':1,'Two':2,'Three':3,'Four':4,'Five':5}

for row in range(len(df)):
  df.loc[row,'rating']=rating_mapping[df.loc[row,'rating']]

df['rating']=pd.to_numeric(df['rating'])

#Converting availability to boolean
df['in_stock']= df['availability']=='In stock'

#Creating GBP to INR price column with 1 GBP = 105.50 INR currency conversion 
df['price_inr']=df['price_gbp']*105.50

##################################################################################################################################
#creating normalized SQLite schema
import sqlite3

con=sqlite3.connect('testdb')
cursor=con.cursor()
categories_query='create table if not exists categories(category_id INTEGER primary key, category_name TEXT UNIQUE)'
cursor.execute(categories_query)
con.commit()

books_query='create table if not exists books(book_id integer primary key, title TEXT, price_gbp REAL, price_inr REAL, rating INTEGER, in_stock INTEGER, category_id INTEGER references categories(category_id))'
cursor.execute(books_query)
con.commit()

#inserting data to categories table

categories={category:index for index,category in enumerate(list(set(df['category'])))}
categories_data=[(index,category) for category,index in categories.items()]
categories_df=pd.DataFrame(categories_data,columns=['category_id','category_name'])
categories_df.to_sql('categories',con,if_exists='append',index=False)

#inserting data to books table.
books_data=[]
for i in range(len(df)):
  books_data.append([i,df.loc[i,'name'],df.loc[i,'price_gbp'],df.loc[i,'price_inr'],df.loc[i,'rating'],df.loc[i,'in_stock'],categories[df.loc[i,'category']]])
books_df=pd.DataFrame(books_data,columns=['book_id','title','price_gbp','price_inr','rating','in_stock','category_id'])
books_df.to_sql('books',con,if_exists='append',index=False)

##################################################################################################################################
## 5 SQL queries using select,where,order by, limit, distinct and in, between with a join

#list count of books under each category
query1='select distinct category_name, count(*) from categories c, books b where c.category_id=b.category_id group by c.category_name'

#list all the books data where category is Historical
query2='select b.* from categories c, books b where c.category_id=b.category_id and c.category_name like "Historical"'

#list 10 highest-rated books one per category and total count of books in that category
query3='select c.category_name, b.title as book_name, count(*) as total_books_under_category from categories c, books b where c.category_id=b.category_id group by c.category_name order by b.rating desc limit 10'

#list distinct book names with rating >4, under category Spirituality
query4='select b.title as book_name ,b.rating,c.category_name from books b, categories c where b.category_id=c.category_id and b.rating>=4 and c.category_name="Spirituality"'

# lowest rated books in the category of Politics,Crime
query5='select distinct b.title as book_name, b.rating from books b, categories c where b.category_id=c.category_id and c.category_name in ("Politics","Crime") and b.rating between 1 and 3'

result1=pd.read_sql(query1,con)
result2=pd.read_sql(query2,con)
result3=pd.read_sql(query3,con)
result4=pd.read_sql(query4,con)
result5=pd.read_sql(query5,con)
##################################################################################################################################

#list count of books under each category
query1='select distinct category_name, count(*) from categories c, books b where c.category_id=b.category_id group by c.category_name'
result1=pd.read_sql(query1,con)
print(result1)

result_df1=categories_df.merge(books_df,how='inner',on='category_id').groupby("category_name")['category_id'].size().reset_index(name='count(*)')
print(result_df1)

#list all the books data where category is Historical
query2='select b.* from categories c, books b where c.category_id=b.category_id and c.category_name like "Historical"'
result2=pd.read_sql(query2,con)
print(result2)

result_df2=books_df.merge(categories_df,how='inner',on='category_id')
mask=result_df2['category_name']=='Historical'
print(result_df2[mask].reset_index())
##################################################################################################################################